# Data Modelling


In [16]:
import pandas as pd
import duckdb
import os
import glob
import magic_duckdb

DATA_DIR = '../data/sap-o2c-data'

In [17]:
con = duckdb.connect("o2c_data.db")

# This ensures it uses '%%sql' and '%sql' instead of its default '%dql'
magic_duckdb.MAGIC_NAME = "sql"
%load_ext magic_duckdb

# THE KEY: Bind it to your existing connection once
%sql -co con

# Configure magic_duckdb to show full results without collapsing
%config MagicDuckDB.displaylimit = 0  # 0 means no limit

The magic_duckdb extension is already loaded. To reload it, use:
  %reload_ext magic_duckdb


## Schema Inference and DDL Generation
We use pandas to read a sample of each dataset to infer the schema and map it to DuckDB types. We store the schema to ensure robust data cleaning during ingestion.

In [18]:
def map_dtype(col):
    dtype = col.dtype
    if pd.api.types.is_integer_dtype(dtype):
        return "BIGINT"
    elif pd.api.types.is_float_dtype(dtype):
        return "DOUBLE"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    else:
        return "VARCHAR"



ddls = {}
for table_name in os.listdir(DATA_DIR):
    table_path = os.path.join(DATA_DIR, table_name)
    if os.path.isdir(table_path):
        jsonl_files = glob.glob(os.path.join(table_path, "*.jsonl"))
        if not jsonl_files:
            continue
        
        # Read sample with pandas
        sample_df = pd.read_json(jsonl_files[0], lines=True, nrows=500)
        
        columns_with_types = []
        for col in sample_df.columns:
            duckdb_type = map_dtype(sample_df[col])
            columns_with_types.append({"name": col, "type": duckdb_type})
        
        column_defs = [f'"{c["name"]}" {c["type"]}' for c in columns_with_types]
        ddl = f"CREATE TABLE IF NOT EXISTS {table_name} (\n    " + ",\n    ".join(column_defs) + "\n);"
        ddls[table_name] = {"ddl": ddl, "columns": columns_with_types}
        
        # Create the table
        con.execute(ddl)
        print(f"Created table: {table_name}")

print("\nAll tables initialized in DuckDB.")

Created table: outbound_delivery_items
Created table: products
Created table: plants
Created table: sales_order_items
Created table: product_plants
Created table: billing_document_headers
Created table: outbound_delivery_headers
Created table: business_partner_addresses
Created table: journal_entry_items_accounts_receivable
Created table: product_storage_locations
Created table: billing_document_cancellations
Created table: business_partners
Created table: payments_accounts_receivable
Created table: sales_order_schedule_lines
Created table: billing_document_items
Created table: customer_company_assignments
Created table: product_descriptions
Created table: sales_order_headers
Created table: customer_sales_area_assignments

All tables initialized in DuckDB.


## Loading Data into DuckDB
Now we load all JSONL files for each table into the corresponding DuckDB tables. We force all columns to `VARCHAR` during initial read using the `columns` parameter, allowing us to perform robust data cleaning (TRIM, NULLIF) and safe casting (TRY_CAST) to maintain data integrity and prevent conversion errors from empty strings.

In [19]:
for table_name, table_info in ddls.items():
    path_pattern = os.path.join(DATA_DIR, table_name, "*.jsonl")
    
    # Generate columns={name: 'VARCHAR', ...} to force all fields to strings during initial read
    columns_params = ", ".join([f'"{c["name"]}": \'VARCHAR\'' for c in table_info["columns"]])
    
    # Robust SELECT clause with TRIM, NULLIF, and TRY_CAST
    select_clauses = []
    for col in table_info["columns"]:
        col_name = col["name"]
        col_type = col["type"]
        
        # 1. TRIM whitespace
        # 2. NULLIF to convert empty strings (after trim) to NULL
        # 3. TRY_CAST to convert to the target type safely
        if col_type == "VARCHAR":
            expr = f'NULLIF(TRIM("{col_name}"), \'\')'
        else:
            expr = f'TRY_CAST(NULLIF(TRIM("{col_name}"), \'\') AS {col_type})'
            
        select_clauses.append(f'{expr} AS "{col_name}"')
    
    select_clause = ", ".join(select_clauses)
    
    # Use columns={...} to skip auto-detection and handle inconsistent types
    query = f"INSERT INTO {table_name} SELECT {select_clause} FROM read_json_auto(\'{path_pattern}\', columns={{{columns_params}}})"
    con.execute(query)
    print(f"Loaded data into {table_name}")

Loaded data into outbound_delivery_items
Loaded data into products
Loaded data into plants
Loaded data into sales_order_items
Loaded data into product_plants
Loaded data into billing_document_headers
Loaded data into outbound_delivery_headers
Loaded data into business_partner_addresses
Loaded data into journal_entry_items_accounts_receivable
Loaded data into product_storage_locations
Loaded data into billing_document_cancellations
Loaded data into business_partners
Loaded data into payments_accounts_receivable
Loaded data into sales_order_schedule_lines
Loaded data into billing_document_items
Loaded data into customer_company_assignments
Loaded data into product_descriptions
Loaded data into sales_order_headers
Loaded data into customer_sales_area_assignments


In [21]:
%%sql 

select * from outbound_delivery_items


,actualDeliveryQuantity,batch,deliveryDocument,deliveryDocumentItem,deliveryQuantityUnit,itemBillingBlockReason,lastChangeDate,plant,referenceSdDocument,referenceSdDocumentItem,storageLocation
0,1,NaN,80738076,10,PC,None,NaN,WB05,740556,10,5031
1,1,ABCD,80738076,20,PC,None,NaN,WB05,740556,20,5031
2,1,ABCD,80738076,30,PC,None,NaN,WB05,740556,30,5031
3,1,ABCD,80738076,40,PC,None,NaN,WB05,740556,40,5031
4,1,ABCD,80738076,50,PC,None,NaN,WB05,740556,50,5031
...,...,...,...,...,...,...,...,...,...,...,...
132,1,ABCD,80754604,10,PC,None,NaN,1301,740539,10,5095
133,1,ABCD,80754605,10,PC,None,NaN,1301,740541,10,5095
134,1,NaN,80754606,10,PC,None,NaN,1301,740542,10,5095
135,2,ABCD,80754606,20,PC,None,NaN,1301,740542,20,5095
